# Admet SDK — UX review

This notebook demonstrates the new `Admet` class for `deeporigin.admet-properties`
(admet-now). Compare with legacy `Molprops` (`deeporigin.mol-props-combined`).

**Run modes**

- **Local mock:** set `DO_ENV=local` and start the mock server (`make mock-server`).
- **Dev:** set `DO_ENV=dev` after `deeporigin login`.

Toggle `TARGET_ENV` below and re-run all cells.

In [ ]:
import os

# Toggle between "local" and "dev"
TARGET_ENV = os.environ.get("DO_ENV", "local")
os.environ["DO_ENV"] = TARGET_ENV


from deeporigin.platform.client import DeepOriginClient

client = DeepOriginClient.from_disk(TARGET_ENV) if TARGET_ENV != "local" else DeepOriginClient()
print(f"env={client.env}")
client

In [ ]:
from deeporigin.drug_discovery import Admet, Ligand, Molprops
from deeporigin.platform.client import DeepOriginClient
client = DeepOriginClient()
client

## 1. Admet — quote then run

`Admet.run(quote=True)` requests a platform estimate without running inference.
`Admet.run()` returns a **DataFrame** and does **not** mutate ligands.

In [ ]:
ligand = Ligand.from_smiles("CCO")
print("ligand.id before:", ligand.id)

props = ["hERG_classification", "AMES_classification", "PPB_regression"]
job = Admet(ligands=[ligand], client=client)

quoted = job.run(quote=True)
print("status:", job.status, "estimate:", job.estimate)
print("ligand.id after quote:", ligand.id)
quoted

In [ ]:
df = job.run()
print("ligand.id after run:", ligand.id)
df

## 2. Contrast with Molprops

`Molprops` uses short property keys (`logp`, `herg`, …) and **mutates** ligands in place.
`Admet` uses admet-now folder names (`hERG_classification`, …) and returns tabular output.

In [ ]:
mp_ligand = Ligand.from_smiles("CCO")
Molprops(ligands=[mp_ligand], props=["logp", "herg"], client=client).run()
print("Molprops mutates ligand attributes:")
print("  log_p:", mp_ligand.log_p)
print("  herg_inhibition_probability:", mp_ligand.herg_inhibition_probability)